# 05 — Comparative Analysis: Germany vs NetherlandsCompare motorway accident safety between:- 🇩🇪 **Unlimited Autobahn** (no speed limit sections)- 🇩🇪 **Speed-limited Autobahn** (sections with 100/120/130 km/h limit)- 🇳🇱 **Dutch motorways** (autosnelwegen, 100/130 km/h)Main metric: **fatalities and accidents per billion vehicle-km** (where vehicle-km data available),otherwise absolute counts normalised by year for trend comparison.

In [ ]:
from pathlib import Pathimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsimport syssys.path.insert(0, '../src')from autobahn_safety.analysis import compute_accident_rate, compare_rates, severity_indexfrom autobahn_safety.visualization import plot_rate_trend, plot_rate_comparisonDATA_RAW  = Path('../data/raw')DATA_PROC = Path('../data/processed')sns.set_theme(style='whitegrid')print('Ready.')

## 1. Load processed data from notebooks 03 and 04

In [ ]:
# Germany — classified accidents (from notebook 03)classified_path = DATA_PROC / 'unfallatlas_classified.parquet'if classified_path.exists():    df_de = pd.read_parquet(classified_path)    print(f'Germany: {len(df_de):,} accident records loaded')else:    print('Run notebook 03 first to generate classified data.')    df_de = None# Netherlands — motorway annual summary (from notebook 04)nl_path = DATA_PROC / 'rws_motorway_annual.parquet'if nl_path.exists():    df_nl_annual = pd.read_parquet(nl_path)    print(f'Netherlands: {len(df_nl_annual)} years loaded')else:    print('Run notebook 04 first to generate NL data.')    df_nl_annual = None

## 2. Severity index comparison (fatalities per accident)

In [ ]:
if df_de is not None:    de_groups = {        'DE: Unlimited Autobahn': df_de[df_de['on_unlimited']],        'DE: Limited Autobahn': df_de[df_de['on_limited_mw']],    }    de_annual = []    for label, grp in de_groups.items():        by_year = grp.groupby('year').agg(            total=('severity', 'count'),            fatal=('severity', lambda x: (x == 'fatal').sum()),        ).reset_index()        by_year['group'] = label        by_year['severity_idx'] = by_year['fatal'] / by_year['total']        de_annual.append(by_year)    df_de_annual = pd.concat(de_annual)    if df_nl_annual is not None:        df_nl_plot = df_nl_annual[['year', 'total', 'fatal', 'severity_idx']].copy()        df_nl_plot['group'] = 'NL: Motorway'        df_combined = pd.concat([df_de_annual, df_nl_plot], ignore_index=True)        fig, ax = plt.subplots(figsize=(11, 5))        palette = {            'DE: Unlimited Autobahn': '#e63946',            'DE: Limited Autobahn': '#457b9d',            'NL: Motorway': '#2a9d8f',        }        for grp_name, grp in df_combined.groupby('group'):            ax.plot(grp['year'], grp['severity_idx'] * 1000,                    marker='o', label=grp_name, linewidth=2,                    color=palette.get(grp_name, 'grey'))        ax.set_title('Severity index: fatalities per 1000 accidents', fontsize=14, fontweight='bold')        ax.set_xlabel('Year'); ax.set_ylabel('Fatalities per 1000 accidents')        ax.legend(); plt.tight_layout(); plt.show()

## 3. Statistical comparison: unlimited vs limited Autobahn

In [ ]:
if df_de is not None:    years = sorted(df_de['year'].unique())    results = []    for yr in years:        yr_data = df_de[df_de['year'] == yr]        n_unl = (yr_data['on_unlimited'] & (yr_data['severity'] == 'fatal')).sum()        n_lim = (yr_data['on_limited_mw'] & (yr_data['severity'] == 'fatal')).sum()        total_unl = yr_data['on_unlimited'].sum()        total_lim = yr_data['on_limited_mw'].sum()        if total_unl > 0 and total_lim > 0:            si_unl = n_unl / total_unl            si_lim = n_lim / total_lim            ratio = si_unl / si_lim if si_lim > 0 else np.nan            results.append({'year': yr, 'fatal_rate_unlimited': si_unl * 1000,                            'fatal_rate_limited': si_lim * 1000, 'rate_ratio': ratio,                            'n_fatal_unlimited': n_unl, 'n_fatal_limited': n_lim})    df_results = pd.DataFrame(results)    print('Fatality rate (per 1000 accidents) — unlimited vs limited Autobahn:')    print(df_results.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
if 'df_results' in dir() and len(df_results):    # Aggregate test across all years    total_unl_fatal = df_results['n_fatal_unlimited'].sum()    total_lim_fatal = df_results['n_fatal_limited'].sum()    total_unl_acc = df_de[df_de['on_unlimited']].shape[0]    total_lim_acc = df_de[df_de['on_limited_mw']].shape[0]    rate_unl = total_unl_fatal / total_unl_acc * 1000    rate_lim = total_lim_fatal / total_lim_acc * 1000    result = stats.poisson_means_test(total_unl_fatal, total_unl_acc, total_lim_fatal, total_lim_acc)    print(f'Fatality rate (per 1000 accidents):')    print(f'  Unlimited Autobahn : {rate_unl:.3f}')    print(f'  Limited Autobahn   : {rate_lim:.3f}')    print(f'  Rate ratio         : {rate_unl/rate_lim:.3f}')    print(f'  Poisson p-value    : {result.pvalue:.4f}')    print()    significance = 'SIGNIFICANT' if result.pvalue < 0.05 else 'NOT significant'    print(f'=> Difference is {significance} (alpha=0.05)')

## 4. Summary bar chart

In [ ]:
if df_de is not None and df_nl_annual is not None:    # Mean severity index per group (overlapping years only)    common_years = set(df_de['year'].unique()) & set(df_nl_annual['year'].unique())    means = {        'DE: Unlimited\nAutobahn': df_de[df_de['on_unlimited'] & df_de['year'].isin(common_years)].apply(            lambda df: (df['severity'] == 'fatal').mean(), axis=0) if False else  # placeholder            (df_de[df_de['on_unlimited'] & df_de['year'].isin(common_years)]['severity'] == 'fatal').mean() * 1000,        'DE: Limited\nAutobahn': (df_de[df_de['on_limited_mw'] & df_de['year'].isin(common_years)]['severity'] == 'fatal').mean() * 1000,        'NL: Motorway': df_nl_annual[df_nl_annual['year'].isin(common_years)]['severity_idx'].mean() * 1000,    }    colors = ['#e63946', '#457b9d', '#2a9d8f']    fig, ax = plt.subplots(figsize=(8, 5))    bars = ax.bar(list(means.keys()), list(means.values()), color=colors, width=0.5, edgecolor='white')    ax.bar_label(bars, fmt='%.2f', padding=4, fontsize=11)    ax.set_title('Mean fatality rate comparison (per 1000 accidents)', fontsize=13, fontweight='bold')    ax.set_ylabel('Fatalities per 1000 accidents')    ax.set_ylim(0, max(means.values()) * 1.3)    sns.despine()    plt.tight_layout(); plt.show()    print('\nConclusion: see values above.')

## 5. Notes on methodology- **Motorway classification (Germany):** spatial join of GPS coordinates with OSM motorway network (50m buffer). Unlimited sections = no `maxspeed` tag or `maxspeed=DE:motorway`.- **Motorway classification (NL):** direct field `MAXSNELHD ∈ {100, 120, 130}` from BRON/RWS dataset.- **Rate metric:** fatalities per 1000 accidents (severity index). Ideally normalised per vehicle-km, but Dutch vehicle-km breakdown by road type is not available in CBS OData for the full period.- **Destatis Autobahn trend** (1979–2021) shows the long-term context; see notebook 03.- **Limitations:** GPS accuracy of Unfallatlas, OSM completeness, different police reporting thresholds between DE and NL.